<a href="https://colab.research.google.com/github/will-genius/C026-01-0727-2023-Wilkister-Kawira/blob/main/The_Knowledge_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import re
import time
import os


In [2]:
# Clone repo
!git clone https://github.com/Kalebu/kamusi.git

# Enter folder
%cd kamusi


Cloning into 'kamusi'...
remote: Enumerating objects: 16, done.
remote: Total 16 (delta 0), reused 0 (delta 0), pack-reused 16 (from 1)
Receiving objects: 100% (16/16), 1.30 MiB | 3.47 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/kamusi


In [3]:
!pip install pdfplumber
import pdfplumber

text = ""

with pdfplumber.open("/content/tuki-raw.pdf") as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

with open("/content/tuki-raw.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 97.4 MB/s eta 0:00:00
Done!


In [4]:
!wget https://huggingface.co/datasets/ngusadeep/Swahili-Corpus-Dataset/resolve/main/Swahili_Corpus_combined.txt


--2026-06-10 12:03:54--  https://huggingface.co/datasets/ngusadeep/Swahili-Corpus-Dataset/resolve/main/Swahili_Corpus_combined.txt
Resolving huggingface.co (huggingface.co)... 13.249.126.128, 13.249.126.91, 13.249.126.122, ...
Connecting to huggingface.co (huggingface.co)|13.249.126.128|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/695c2b9fde103ec72a04885f/24127c61deff7c4a8283ec127a4410773efb8f8bbd20287ec3b128058f3ad1dd?Expires=1781096634&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY5NWMyYjlmZGUxMDNlYzcyYTA0ODg1Zi8yNDEyN2M2MWRlZmY3YzRhODI4M2VjMTI3YTQ0MTA3NzNlZmI4ZjhiYmQyMDI4N2VjM2IxMjgwNThmM2FkMWRkKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MTA5NjYzNH19fV19&Signature=MEQCICDRMN8VsBVru-LjzN3JbVmbdAaEKSGiabW9giEh2UfvAiBTuxfeWxvsuf9MMIErXIDvVf80ITAR57uh1jy-9xZIFA__&Key-Pair-Id=K1LYXO563TGWFU&X-Xet-Cas-Uid=public&response-conten

In [14]:
INPUT_DIR = "/content"
KALEBU_PATH = os.path.join(INPUT_DIR, "kamusi", "words.json")
TUKI_PATH = os.path.join(INPUT_DIR, "tuki-raw.txt")
CORPUS_PATH = os.path.join(INPUT_DIR, "Swahili_Corpus_combined.txt")
OUTPUT_PATH = "/content/perfect_bilingual_knowledge_base.json"

In [15]:
def clean_text(text):
    if not text:
        return ""
    return re.sub(r'\s+', ' ', text.strip())

In [16]:
def clean_english_keywords(raw_line, raw_swahili_word):
    """
    Applies strict structural pattern rules to extract pure English keywords
    without part-of-speech tags, numbers, or example sentences.
    """
    # 1. Remove the leading Swahili word token from the processing line
    working_text = raw_line.replace(raw_swahili_word, "", 1).strip()

    # 2. Isolate main definitions by cutting away usage sentence text blocks
    if ":" in working_text:
        working_text = working_text.split(":")[0].strip()

    # 3. Discard grammatical structural plural brackets if present
    if "]" in working_text:
        working_text = working_text.split("]")[-1].strip()
    else:
        # If no brackets exist, drop the leading Part of Speech token
        working_text = re.sub(r'^(nm|kt|kv|kl|ki)\s+', '', working_text).strip()

    # 4. If secondary grammatical flags cut into definitions, drop the trailing portion
    for marker in [' nm ', ' kt ', ' kv ', ' kl ', ' ki ']:
        if marker in working_text:
            working_text = working_text.split(marker)[0].strip()

    # 5. Clear out language source metadata parentheticals and formatting digits
    working_text = re.sub(r'\(.*?\)', '', working_text)
    working_text = re.sub(r'\d+\s+', '', working_text)

    # 6. Segment definition fragments by punctuation markers
    meaning_candidates = re.split(r'[\.,;]', working_text)

    clean_keywords = []
    for candidate in meaning_candidates:
        clean_item = candidate.strip().lower()
        # Filter out empty string cuts or formatting noise terms
        if clean_item and len(clean_item) > 1 and not clean_item.startswith('pia'):
            clean_keywords.append(clean_item)

    # 7. Restrict target terms to the top 2 distinct direct keyword results
    final_keywords = clean_keywords[:2]
    if final_keywords:
        return " | ".join(final_keywords)

    return "swahili vocabulary entry"

In [17]:
def normalize_tuki_word(raw_word_token):
    """
    Cleans raw TUKI dictionary structural keys by stripping suffix markers,
    structural metadata symbols, and trailing homonym digits.

    """
    # Remove symbols (*, !) and internal suffix separation periods (.)
    clean = raw_word_token.replace('*', '').replace('!', '').replace('.', '')
    # Strip any trailing homonym identification numbers at the end of the word block
    clean = re.sub(r'\d+$', '', clean)
    return clean.strip().lower()

In [18]:
def parse_tuki_file(filepath):
    # This lookup map will now store lists of dictionaries to hold multiple meanings per word
    tuki_map = {}
    pos_labels = {
        'nm': 'nomino (noun)', 'kt': 'kitenzi (verb)',
        'kv': 'kivumishi (adjective)', 'kl': 'kielezi (adverb)', 'ki': 'kiingishi (interjection)'
    }

    if not os.path.exists(filepath):
        print(f"Critical Error: TUKI dataset file missing at '{filepath}'.")
        return None

    with open(filepath, "r", encoding="utf-8") as f:
        raw_lines = f.readlines()

    for line in raw_lines:
        if not line.strip() or len(line.split()) < 2:
            continue

        raw_word = line.split()[0]

        clean_word = normalize_tuki_word(raw_word)

        part_of_speech = "haijulikani "
        for label in pos_labels.keys():
            if f" {label} " in line:
                part_of_speech = pos_labels[label]
                break

        content_block = line.replace(raw_word, "", 1).strip()
        full_definition_english = content_block.replace("(Kar)", "").replace("(Kng)", "").strip()

        # Run our strict pattern extractor from the previous step
        clean_word_english = clean_english_keywords(line, raw_word)

        entry_payload = {
            "english_word_match": clean_word_english,
            "part_of_speech": part_of_speech,
            "definition_english": full_definition_english,
            "processed": False
        }

        # If the word is a homonym and already exists, append this new meaning instance to the list
        if clean_word in tuki_map:
            tuki_map[clean_word].append(entry_payload)
        else:
            # First time seeing this word, initialize it as a list holding our payload dictionary
            tuki_map[clean_word] = [entry_payload]

    return tuki_map

In [19]:
def load_swahili_corpus(filepath):
    if not os.path.exists(filepath):
        print(f"Critical Error: Swahili sentence corpus dataset missing at '{filepath}'.")
        return None
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f.readlines() if line.strip()]

In [20]:
def find_corpus_example(word, corpus_lines):
    clean_query = word.replace('!', '').strip().lower()
    for line in corpus_lines:
        if clean_query in line.lower():
            return [line]
    return []

In [21]:
def run_algorithmic_translation(word, definition_sw):
    def_low = definition_sw.lower()
    if "mtu wa thamani" in def_low or "mwandani" in def_low:
        return "precious | cherished person", "Refers to a highly valued person or close confidant."
    if "sauti" in def_low:
        return "exclamation", f"An exclamation sound matching context: {definition_sw}"
    return "swahili vocabulary entry", f"Translated definition matching context: {definition_sw}"

In [22]:
def execute_master_union_pipeline(max_entries):
    print(" Initializing Fixed Pipeline Architecture...")
    if not os.path.exists(INPUT_DIR):
        print(f"Critical Error: The base directory '{INPUT_DIR}' does not exist.")
        print(f"Please create a folder named '{INPUT_DIR}' in your project root manually.")
        return
    # Process files or abort if dependencies are absent
    tuki_lookup = parse_tuki_file(TUKI_PATH)
    if tuki_lookup is None:
        return

    corpus_source = load_swahili_corpus(CORPUS_PATH)
    if corpus_source is None:
        return

    if not os.path.exists(KALEBU_PATH):
        print(f" Critical Error: Kalebu words.json repository file missing at '{KALEBU_PATH}'.")
        return

    with open(KALEBU_PATH, "r", encoding="utf-8") as f:
        kalebu_data = json.load(f)

    final_records = []
    global_counter = 0

    print("\n Running Pass 1: Mapping Kalebu database files to TUKI definitions...")

    # Track words we have already processed from Kalebu to guarantee 100% uniqueness
    seen_kalebu_words = set()

    for key, item in kalebu_data.items():
        if global_counter >= max_entries:
            break

        word_sw = clean_text(item.get("Word", ""))
        meaning_sw = clean_text(item.get("Meaning", ""))
        synonyms = item.get("Synonyms", "")

        lookup_key = word_sw.replace('!', '').replace('*', '').strip().lower()

        # GUARDRAIL: If we already built a consolidated entry for this word, skip duplicate rows
        if lookup_key in seen_kalebu_words:
            continue

        seen_kalebu_words.add(lookup_key)

        # Cross-reference TUKI dataset map
        if lookup_key in tuki_lookup and len(tuki_lookup[lookup_key]) > 0:
            tuki_instances = tuki_lookup[lookup_key]

            # Collapse multiple English keyword strings cleanly
            word_en = " | ".join([inst["english_word_match"] for inst in tuki_instances])

            # Deduplicate and combine distinct Parts of Speech tags
            pos_set = list(set([inst["part_of_speech"] for inst in tuki_instances]))
            pos = " | ".join(pos_set)

            # Build an explicit numbered list item string for the detailed english block
            def_en_parts = [f"Meaning {idx+1}: {inst['definition_english']}" for idx, inst in enumerate(tuki_instances)]
            definition_en = " | ".join(def_en_parts)

            # Mark all matching TUKI sub-instances as processed so Pass 2 skips them
            for inst in tuki_instances:
                inst["processed"] = True

            layer = "tuki_collapsed_homonym_alignment"
        else:
            # Fallback layer if word is missing inside TUKI
            word_en, definition_en = run_algorithmic_translation(lookup_key, meaning_sw)
            pos = "nomino (noun)" if "mtu" in meaning_sw.lower() else "haijulikani (unspecified)"
            layer = "algorithmic_translation_fallback"

        examples = find_corpus_example(lookup_key, corpus_source)
        syn_clean = synonyms if synonyms and synonyms != "None" else "None"
        vector_text = f"Swahili Word: {word_sw} ({word_en}) | Type: {pos} | Definition: {meaning_sw} | Synonyms: {syn_clean}"

        global_counter += 1
        final_records.append({
            "word_id": f"sw_en_{str(global_counter).zfill(6)}",
            "word_swahili": word_sw,
            "word_english": word_en,
            "part_of_speech": pos,
            "definition_swahili": meaning_sw,
            "definition_english": definition_en,
            "synonyms_swahili": synonyms if synonyms else None,
            "usage_examples_swahili": examples,
            "vector_input_text": vector_text,
            "metadata": {"source": "kalebu_base", "validation_layer": layer}
        })

    print(" Running Pass 2: Capturing unique vocabulary entries unique to TUKI...")
    for tuki_word, instances_list in tuki_lookup.items():
        if global_counter >= max_entries:
            print(f" Reached absolute limit of {max_entries} words. Stopping data collection.")
            break

        # Filter out instances: look for items where ALL homonym matches are completely unprocessed
        unprocessed_instances = [inst for inst in instances_list if not inst["processed"]]

        if unprocessed_instances:
            # Collapse them into a standalone entry format
            word_en = " | ".join([inst["english_word_match"] for inst in unprocessed_instances])
            pos = " | ".join(list(set([inst["part_of_speech"] for inst in unprocessed_instances])))
            def_en = " | ".join([inst["definition_english"] for inst in unprocessed_instances])

            examples = find_corpus_example(tuki_word, corpus_source)
            vector_text = f"Swahili Word: {tuki_word} ({word_en}) | Type: {pos} | Definition: {def_en}"

            global_counter += 1
            final_records.append({
                "word_id": f"sw_en_{str(global_counter).zfill(6)}",
                "word_swahili": tuki_word,
                "word_english": word_en,
                "part_of_speech": pos,
                "definition_swahili": "Maana haikuandikwa kwenye kamusi ya Kalebu",
                "definition_english": f"Refers to a contextual definition of: {def_en}",
                "synonyms_swahili": None,
                "usage_examples_swahili": examples,
                "vector_input_text": vector_text,
                "metadata": {"source": "tuki_base", "validation_layer": "tuki_standalone_entry"}
            })

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(final_records, f, indent=2, ensure_ascii=False)

    print(f"\n==================================================")
    print(f" Complete Bilingual Master Union Generated Successfully!")
    print(f"File saved to path location: '{OUTPUT_PATH}'")
    print(f"Processed {len(final_records)} pristine entries into the vector schema.")

if __name__ == "__main__":
    execute_master_union_pipeline(20)

 Initializing Fixed Pipeline Architecture...

 Running Pass 1: Mapping Kalebu database files to TUKI definitions...
 Running Pass 2: Capturing unique vocabulary entries unique to TUKI...
 Reached absolute limit of 20 words. Stopping data collection.

 Complete Bilingual Master Union Generated Successfully!
File saved to path location: '/content/perfect_bilingual_knowledge_base.json'
Processed 20 pristine entries into the vector schema.
